In [ ]:
!pip install -U pageindex openai python-dotenv

In [ ]:
import os, json, time
from dotenv import load_dotenv
load_dotenv()
PAGEINDEX_API_KEY = ""
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")
print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("OpenAI key loaded:   ", "✅" if OPENAI_API_KEY    else "❌ Missing!")

In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI
pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

In [ ]:
PDF_PATH = "./sample_document.pdf"
print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]
print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

In [ ]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")
while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    time.sleep(5)

In [ ]:
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])
print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

In [ ]:
def print_tree(nodes, indent=0):
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)
print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

In [ ]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total
total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

In [ ]:
def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    compressed_tree = compress(tree)
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.
Query: {query}
Document Tree:
{json.dumps(compressed_tree, indent=2)}
Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
query = "What is the syllabus covered in Modern LLM finetuning?"
print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)
print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

In [ ]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [ ]:
def generate_answer(query: str, nodes: list, model: str = "gpt-4o") -> str:
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.
Question: {query}
Context:
{context}
Answer:"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    nodes = find_nodes_by_ids(tree, node_ids)
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    answer = generate_answer(query, nodes)
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    return answer

In [ ]:
answer = vectorless_rag(
    query="What are the syllabus covered in modern llm finetuning?",
    tree=pageindex_tree
)

In [ ]:
test_queries = [
    "What are the syllabus covered in modern llm finetuning?",
    "What are the syllabus covered in RAG?",
    "Summarize the syllabus of Tokenization Deep Dive?",
]
for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)

In [ ]:
FINANCIAL_EXPERT_RULES = """
Expert routing rules for financial documents (10-K, annual reports):
- EBITDA, profitability queries    → MD&A section (Management Discussion & Analysis)
- Liquidity, cash flow queries     → Cash Flow Statement + liquidity footnotes
- Risk factor queries              → Part I, Item 1A (Risk Factors)  
- Revenue breakdown queries        → Segment reporting or Item 7
- Forward-looking / strategy       → CEO letter, Outlook, Strategy section
- Debt, credit, leverage queries   → Balance Sheet + debt footnotes
- Regulatory / compliance queries  → Legal Proceedings or regulatory filings

In [ ]:
FINANCIAL_EXPERT_RULES = """
Route queries to the correct module using these rules:
M1  Neural Network Refresher   → backprop, activations, optimizers, PyTorch basics
M2  Hardware                   → GPU, TPU, Apple Silicon, compute infrastructure
M3  Transformers 101           → attention, self-attention, encoder-decoder, MHA
M4  Tokenization               → BPE, WordPiece, SentencePiece, Byte Latent Transformers
M5  Finetuning Architectures   → hands-on BERT/GPT/T5 finetuning, Hugging Face
M6  KV Cache & Attention       → KV cache, Flash Attention, MQA, GQA, RoPE, vLLM
M7  Scaling Laws               → Kaplan, Chinchilla, compute-optimal training
M8  Mixture of Experts         → MoE, sparse computation, Mixture of Depths
M9  Modern LLM Finetuning      → LoRA, QLoRA, SFT, DPO, PPO, RLHF, GRPO, ORPO,
                                  quantization, TRL, Unsloth, synthetic data,
                                  reasoning models, evaluation, deployment
M10 SLM                        → small language models, pruning, when SLM vs LLM
M11 Knowledge Distillation     → student-teacher, soft labels, DistilBERT, DeepSeek-R1
M12 Hybrid Architectures       → Mamba, RWKV, SSMs, Jamba, Nemotron, beyond Transformers
M13 Vision Foundations         → ViT, patch embeddings, CLIP, SigLIP, DINOv2
M14 Visual Language Models     → VLM architecture, aligner, multimodal reasoning
M15 Stable Diffusion & DiT     → DDPM, latent diffusion, FLUX.1, ControlNet, DreamBooth
M16 Embedding Models           → dense, sparse, binary, Matryoshka, MRL, fine-tuning
M17 RAG                        → chunking, BM25, ColBERT, hybrid RAG, rerankers,
                                  self/corrective/adaptive/agentic RAG, Graph RAG,
                                  multi-modal RAG, ColPali, RAG security
M18 Context Engineering        → prompt vs context engineering, memory architecture,
                                  context compression, KV cache, agent context lifecycle
M19 DSPy                       → signatures, modules, MIPROv2, self-optimizing RAG
M20 Agents                     → ReAct, MCP, LangGraph, CrewAI, browser agents,
                                  A2A, guardrails, observability, evaluation
M21 RL                         → PPO, GRPO, DAPO, GSPO, CISPO, reward models,
                                  RLHF vs RLVR, policy gradient, DeepSeek-R1 training
Cross-cutting rules:
- "learning path / where to start"     → M1 → M2 → M3 in order
- "production / deployment / serving"  → M9 (quantization) + M20 (agents)
- "fine-tuning vs RAG"                 → M9 + M17 + M18
- "multimodal / vision + language"     → M13 + M14 + M17 (multi-modal RAG)
- "reasoning models / test-time RL"    → M9 (reasoning) + M21 (GRPO/DAPO)

In [ ]:
def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "gpt-4o"
) -> dict:
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.
Query: {query}
Document Tree:
{json.dumps(compress(tree), indent=2)}
Expert Routing Rules (follow these carefully):
{expert_rules}
Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [ ]:
query = "Details of the modern llm finetuning?"
print(f"🔍 Query: {query}\n")
print("── Without Expert Rules ──")
basic   = llm_tree_search(query, pageindex_tree)
print("Nodes:", basic.get("node_list"))
print()
print("── With Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, FINANCIAL_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", "")[:300])

In [ ]:
def expert_rag(query: str, tree: list, rules: str) -> str:
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)
answer = expert_rag(
    query="Details of the syllabus of modern llm finetuning",
    tree=pageindex_tree,
    rules=FINANCIAL_EXPERT_RULES
)
print(answer)

In [ ]:
question = "What are the key findings in this document?"
response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)
answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

In [ ]:
conversation_history = []
def chat_with_doc(user_message: str, doc_id: str) -> str:
    global conversation_history
    conversation_history.append({"role": "user", "content": user_message})
    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )
    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})
    return assistant_reply
questions = [
    "What were the main revenue sources last year?",
    "How does that compare to the year before?",
    "What factors drove that change?"
]
for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)

In [ ]:
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt

In [ ]:
import os
openai_key = os.getenv("OPENAI_API_KEY", "your_key_here")
with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")
print("✅ .env created for self-hosted mode")

In [ ]:
PDF_PATH = "/path/to/your/document.pdf"
!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

In [ ]:
import json
TREE_JSON_PATH = "/path/to/your/document_pageindex.json"
with open(TREE_JSON_PATH, "r") as f:
    local_tree = json.load(f)
print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

In [ ]:
query  = "Summarize the executive summary section."
answer = vectorless_rag(query, local_tree)

In [ ]:
print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

In [ ]:
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")